# PART 7-2. 감성 분석 - BERT (Hugging Face Transformers)

## 수업 목표
- 이미 대용량 데이터로 사전학습된 BERT 모델을 가져와서 감성분석을 수행합니다.
- PART 7-1(직접 학습)과 PART 7-2(사전학습 모델)의 차이를 이해합니다.

## 수업 진행 포인트
| 구분 | 설명 |
|---|---|
| 데이터 관점 | 같은 리뷰 데이터를 사용합니다. |
| 코드 관점 | Hugging Face pipeline 사용법을 익힙니다. |
| AI 관점 | 직접 학습보다 사전학습 모델이 더 강력한 이유를 이해합니다. |
| 강사 메모 | 한국어도 어느 정도 처리 가능한 다국어 모델을 사용합니다. |




## 📌 이 파트에서 사용하는 API 흐름 (HuggingFace)

| 단계 | 함수 | 역할 |
|---|---|---|
| ① | pipeline() | 모델 로드 + 전처리 + 예측 한번에 |
| ② | predict_bert() | 예측 실행 |

> BERT는 사전학습 모델 → Define / Compile / Fit 없이 바로 사용


## PART 7-1 vs PART 7-2 비교

| 항목 | Part 7-1 (기초) | Part 7-2 (BERT) |
|---|---|---|
| 학습 방식 | 직접 학습 | 사전학습 모델 사용 |
| Tokenizer | 직접 만들기 | 모델 내부 포함 |
| 언어 지원 | 학습 데이터 언어만 | 다국어 (한국어 포함) |
| 성능 | 보통 | 우수 |
| 코드 복잡도 | 높음 | 낮음 |

In [1]:
# 셀 1. Google Drive 연결

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# 셀 2. Hugging Face 라이브러리 설치
# 처음 실행 시에만 설치가 필요합니다.

!pip install transformers torch

#!pip install transformers torch -q
# -q: 설치 로그 줄이기

In [5]:
# 셀 3. 라이브러리 불러오기

import pandas as pd
from transformers import pipeline
# pipeline: 모델 불러오기 + 전처리 + 예측을 한 번에 처리해주는 함수

base_path = "/content/drive/MyDrive/Olist"
print("라이브러리 불러오기 완료!")

라이브러리 불러오기 완료!


In [7]:
# 셀 4. 데이터 불러오기 및 전처리

df = pd.read_csv(f"{base_path}/data/olist_master_data.csv")

review_df = df[["review_comment_message", "review_score"]].copy()
review_df = review_df.dropna()

# 이진 분류(Binary Classification)를 위해 중립(3점) 리뷰는 제거
review_df = review_df[review_df["review_score"] != 3]

review_df["label"] = review_df["review_score"].apply(
    lambda s: 1 if s >= 4 else 0
)

print(f"분석할 리뷰 수: {len(review_df):,}")
review_df.head()

분석할 리뷰 수: 45,714


,review_comment_message,review_score,label
0,"Não testei o produto ainda, mas ele veio corre...",4.0,1
1,"Não testei o produto ainda, mas ele veio corre...",4.0,1
2,"Não testei o produto ainda, mas ele veio corre...",4.0,1
3,Muito bom o produto.,4.0,1
5,O produto foi exatamente o que eu esperava e e...,5.0,1



### 🔵 ① pipeline() → 모델 로드


In [8]:
# 셀 5. BERT 감성분석 모델 불러오기
# 다국어 감성분석 모델 (포르투갈어, 한국어 등 지원)
# 처음 실행 시 모델 파일 다운로드 (약 200MB, 약 30초 소요)

sentiment_model = pipeline(
    "sentiment-analysis",
    model="lxyuan/distilbert-base-multilingual-cased-sentiments-student"
)

print("모델 불러오기 완료!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/759 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/541M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.92M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

모델 불러오기 완료!


In [9]:
# 셀 6. 리뷰 1개로 먼저 테스트

test_text = "배송이 빠르고 상품이 좋아요"
result = sentiment_model(test_text)

print(f"입력: {test_text}")
print(f"결과: {result}")
# ▶ label=POSITIVE/NEGATIVE, score=신뢰도(0~1)

입력: 배송이 빠르고 상품이 좋아요
결과: [{'label': 'positive', 'score': 0.8837409615516663}]



### 🔵 ② predict_bert() → 예측 실행


In [10]:
# 셀 7. 감성분석 함수 만들기

def predict_bert(text):
    """
    BERT 모델로 문장의 감성을 예측합니다.
    텍스트는 최대 512글자까지 처리됩니다.
    """
    result = sentiment_model(text[:512])[0]
    return result["label"], round(result["score"], 3)

# 테스트
print(predict_bert("produto muito bom gostei"))  # 좋은 제품이에요
print(predict_bert("produto ruim nao gostei"))   # 나쁜 제품이에요
print(predict_bert("배송이 빠르고 상품이 좋아요"))

('positive', 0.638)
('negative', 0.5)
('positive', 0.884)


In [11]:
# 셀 8. 샘플 리뷰 20개에 BERT 적용 (약 30초)

sample_df = review_df.sample(20, random_state=42).copy()

sample_df["ai_result"] = sample_df["review_comment_message"].apply(predict_bert)
sample_df["ai_label"]  = sample_df["ai_result"].apply(lambda x: x[0])
sample_df["ai_score"]  = sample_df["ai_result"].apply(lambda x: x[1])

result_df = sample_df[[
    "review_comment_message", "review_score", "label", "ai_label", "ai_score"
]]
result_df

,review_comment_message,review_score,label,ai_label,ai_score
20215,"Entrega muita rápida, produto exatamente igual...",5.0,1,positive,0.841
109171,"A luz é forte, tem uma base sólida e firme, ba...",5.0,1,positive,0.745
71120,Otimo,5.0,1,positive,0.556
29228,Adorei! Obrigada!,5.0,1,positive,0.486
37790,não recebi,1.0,0,neutral,0.743
100591,"prateleira de tamanho ideal ao que eu esperava,",5.0,1,positive,0.860
114062,Bom gostei entregue dentro do prazo e só tenho...,5.0,1,positive,0.773
107700,Comprei 6 e só recebi 5.\r\nJá entrei em conta...,1.0,0,negative,0.591
110684,Falta de respeito e interesse do fornecedor. C...,1.0,0,negative,0.728
19414,adorei este kit,5.0,1,positive,0.637


In [12]:
# 셀 9. 실제 평점 vs AI 예측 비교 출력
# for i in range(10, 15):
for i in range(5):
    row = result_df.iloc[i]
    print(f"\n[리뷰 {i+1}]")
    print(f"  문장:      {str(row['review_comment_message'])[:60]}...")
    print(f"  실제 평점: {row['review_score']}점")
    print(f"  실제 라벨: {'긍정' if row['label']==1 else '부정'}")
    print(f"  AI 예측:   {row['ai_label']}")
    print(f"  AI 신뢰도: {row['ai_score']:.1%}")


[리뷰 1]
  문장:      Entrega muita rápida, produto exatamente igual as imagens....
  실제 평점: 5.0점
  실제 라벨: 긍정
  AI 예측:   positive
  AI 신뢰도: 84.1%

[리뷰 2]
  문장:      A luz é forte, tem uma base sólida e firme, bastante potênci...
  실제 평점: 5.0점
  실제 라벨: 긍정
  AI 예측:   positive
  AI 신뢰도: 74.5%

[리뷰 3]
  문장:      Otimo...
  실제 평점: 5.0점
  실제 라벨: 긍정
  AI 예측:   positive
  AI 신뢰도: 55.6%

[리뷰 4]
  문장:      Adorei! Obrigada!...
  실제 평점: 5.0점
  실제 라벨: 긍정
  AI 예측:   positive
  AI 신뢰도: 48.6%

[리뷰 5]
  문장:      não recebi...
  실제 평점: 1.0점
  실제 라벨: 부정
  AI 예측:   neutral
  AI 신뢰도: 74.3%


In [13]:
# 셀 10. 랜덤 5개 리뷰 출력
sample_df = result_df.sample(5)

for i, row in sample_df.iterrows():

    print(f"\n[리뷰 {i}]")
    print(f"  문장:      {str(row['review_comment_message'])[:60]}...")
    print(f"  실제 평점: {row['review_score']}점")
    print(f"  실제 라벨: {'긍정' if row['label']==1 else '부정'}")
    print(f"  AI 예측:   {row['ai_label']}")
    print(f"  AI 신뢰도: {row['ai_score']:.1%}")


[리뷰 82629]
  문장:      Chegou no prazo....
  실제 평점: 4.0점
  실제 라벨: 긍정
  AI 예측:   neutral
  AI 신뢰도: 42.5%

[리뷰 109171]
  문장:      A luz é forte, tem uma base sólida e firme, bastante potênci...
  실제 평점: 5.0점
  실제 라벨: 긍정
  AI 예측:   positive
  AI 신뢰도: 74.5%

[리뷰 19414]
  문장:      adorei este kit...
  실제 평점: 5.0점
  실제 라벨: 긍정
  AI 예측:   positive
  AI 신뢰도: 63.7%

[리뷰 51291]
  문장:      Pedido entregue incompleto....
  실제 평점: 1.0점
  실제 라벨: 부정
  AI 예측:   negative
  AI 신뢰도: 71.6%

[리뷰 110684]
  문장:      Falta de respeito e interesse do fornecedor. Como pode um pr...
  실제 평점: 1.0점
  실제 라벨: 부정
  AI 예측:   negative
  AI 신뢰도: 72.8%




## 마무리 정리

| 확인할 내용 | 설명 |
|---|---|
| 이 파트의 핵심 | 사전학습 BERT 모델로 리뷰 감성을 분석합니다. |
| 사전학습 모델의 장점 | 직접 학습 없이도 높은 성능, 다국어 지원 |
| 다음 단계 | PART 8에서 매출 금액을 예측합니다. |